# Lecture 3: Distributed Paradigms

**Goal:** Understand why MapReduce-style aggregation fails for deep learning, and learn the collective communication algorithms that power modern GPU clusters.

---

## 1. Hardware Primitives: PCIe, NVLink, and Ethernet

### Why Spark Fails at Parameter Aggregation
Apache Spark was designed for **data parallelism** — distributing data across nodes and running independent computations (map) followed by aggregation (reduce). This works brilliantly for:
- Word count, log analysis, ETL pipelines
- SQL queries over distributed tables
- Feature engineering and batch preprocessing

But deep learning requires **model parallelism** or **data-parallel gradient synchronization**, where every node must exchange **the entire model's gradients** after every minibatch. For a typical model:

| Model | Parameters | Gradient Size (FP32) |
|-------|-----------|---------------------|
| ResNet-50 | 25.6M | 97.7 MB |
| GPT-2 | 1.5B | 5.7 GB |
| GPT-3 | 175B | 700 GB |
| Llama 3 70B | 70B | 280 GB |

These gradients must be exchanged **every 0.5-2 seconds** (per minibatch). Spark's TCP/IP networking can't sustain this.

### The Hardware Stack

**Standard Ethernet (Spark clusters):**
```
GPU → PCIe Bus → CPU → NIC → Ethernet Switch → NIC → CPU → PCIe Bus → GPU
```
- Each hop adds latency (PCIe: ~μs, TCP stack: ~10μs, switch: ~1μs)
- CPU must manage every packet (interrupt-driven or polling)
- Maximum practical throughput: ~12.5 GB/s (100 Gbps Ethernet)
- **CPU is the bottleneck** — it can't process packets fast enough

**InfiniBand / NVLink (AI clusters like NVIDIA DGX):**
```
GPU → NVLink → GPU  (direct, no CPU involvement!)
```
- **RDMA** (Remote Direct Memory Access): GPU writes directly to remote GPU's VRAM
- No CPU interrupts, no TCP stack, no kernel context switches
- NVLink 4.0 bandwidth: 900 GB/s (per GPU!)
- InfiniBand NDR: 400 Gbps per port, kernel-bypass RDMA

### Why This Matters for the Equations Below
The bandwidth $B$ in our equations represents the **effective** per-node bandwidth:
- Ethernet: $B \approx 10$ GB/s (shared across all nodes through a switch)
- InfiniBand: $B \approx 50$ GB/s (per-node dedicated links)
- NVLink: $B \approx 450$ GB/s (within a single server's GPU interconnect)

The communication algorithm determines how efficiently this bandwidth is utilized.

---

## 2. Distributed Strategies: Parameter Server vs Ring-AllReduce

### The Parameter Server (Hub-and-Spoke)
This is the natural extension of Spark's driver-worker pattern:
1. Each worker computes gradients on its local minibatch
2. All workers send their gradients to a central **parameter server**
3. The server averages all gradients and broadcasts the updated model back

**Problem:** The server's NIC must receive $M$ bytes from each of $N$ workers:

$$T_{PS} = \frac{M \cdot N}{B}$$

This scales **linearly** with the number of nodes. At $N=1000$ nodes with a 5 GB model:
$$T_{PS} = \frac{5 \times 1000}{50} = 100 \text{ seconds per iteration!}$$

The bottleneck is the server's single network interface — it's physically impossible to receive data faster than the NIC allows.

---

### Ring-AllReduce (Collective Communication)
**Key Insight:** Instead of funneling all data through one node, arrange $N$ nodes in a logical ring. Every node sends and receives simultaneously.

#### Phase 1: Scatter-Reduce ($N-1$ steps)
1. Divide the gradient vector into $N$ equal chunks
2. In step $i$, each node sends chunk $i$ to its right neighbor and receives chunk $i-1$ from its left neighbor
3. Upon receiving, **add** the received chunk to its local copy (reduce operation)
4. After $N-1$ steps, each node holds the **fully reduced sum** of exactly one chunk

#### Phase 2: All-Gather ($N-1$ steps)
1. In step $i$, each node sends its completed chunk to its right neighbor
2. After $N-1$ steps, every node has the complete, fully-reduced gradient vector

**Total data transferred per node:**
- Each phase: $(N-1)$ steps × $\frac{M}{N}$ bytes per step
- Total: $2 \times (N-1) \times \frac{M}{N}$ bytes

$$T_{Ring} = 2 \times \frac{M(N-1)}{N \cdot B}$$

Taking the limit as $N \to \infty$:
$$\lim_{N \to \infty} T_{Ring} = \lim_{N \to \infty} \frac{2M(N-1)}{NB} = \frac{2M}{B}$$

**This is constant!** Adding more nodes does NOT increase communication time. This is why NVIDIA can train on 30,000 GPUs without the communication becoming a bottleneck.

---

### Bandwidth Optimality
Ring-AllReduce is **bandwidth-optimal**: each node sends exactly $\frac{2M(N-1)}{N}$ bytes, which approaches $2M$ as $N$ grows. No algorithm can do better — you must send at least $M$ bytes out (scatter) and receive at least $M$ bytes (gather), totaling $2M$ bytes of communication per node.

The effective per-node bandwidth utilization is:
$$\text{Utilization}_{Ring} = \frac{N-1}{N} \approx 1 \quad (100\%)$$
$$\text{Utilization}_{PS} = \frac{1}{N} \approx 0 \quad (0\% \text{ as } N \to \infty)$$

In [ ]:
# ============================================================================
# INTERACTIVE DISTRIBUTED TRAINING SIMULATOR
# ============================================================================
# This cell visualizes the fundamental scaling difference between
# Parameter Server (hub-and-spoke) and Ring-AllReduce (collective) topologies.

import numpy as np                       # Numerical arrays
import matplotlib.pyplot as plt          # Plotting
from ipywidgets import interact, IntSlider, FloatSlider, VBox, HBox, Output
import ipywidgets as widgets             # Interactive widgets
from IPython.display import display      # Widget rendering

def plot_advanced_communication(N_max=1000, M=10.0, B=100.0):
    """
    Plots two side-by-side graphs comparing Parameter Server vs Ring-AllReduce:
    1. Communication Time vs Number of Nodes (left)
    2. Effective Bandwidth Utilization vs Number of Nodes (right)
    
    Args:
        N_max (int): Maximum number of nodes to simulate (x-axis upper bound)
        M (float): Model size in gigabytes
        B (float): Per-node network bandwidth in GB/s
    """
    # Generate N values from 2 to N_max (need at least 2 nodes for distributed)
    N_vals = np.linspace(2, N_max, 500)
    
    # === Parameter Server Communication Time ===
    # T_PS = (M × N) / B
    # The central server must receive M bytes from EACH of N workers
    # All traffic funnels through one NIC → linear scaling
    T_PS = (M * N_vals) / B
    
    # === Ring-AllReduce Communication Time ===
    # T_Ring = 2 × M(N-1) / (N × B)
    # Each node sends/receives 2(N-1)/N × M bytes total
    # As N→∞, this approaches 2M/B (constant!)
    T_Ring = 2 * (M * (N_vals - 1)) / (N_vals * B)
    
    # Create side-by-side subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # --- Left Plot: Communication Time ---
    ax1.plot(N_vals, T_PS, label='Parameter Server $T_{PS}$', color='#e74c3c', linewidth=2.5)
    ax1.plot(N_vals, T_Ring, label='Ring-AllReduce $T_{Ring}$', color='#2ecc71', linewidth=2.5)
    
    # Set y-axis to highlight the Ring plateau (cap at 3× the Ring asymptote)
    ring_asymptote = 2 * M / B  # The limiting value as N→∞
    ax1.axhline(ring_asymptote, color='#2ecc71', linestyle=':', alpha=0.5,
                label=f'Ring asymptote: {ring_asymptote:.2f}s')
    ax1.set_ylim(0, max(ring_asymptote * 3, 2))
    ax1.set_xlim(2, N_max)
    ax1.set_xlabel('Number of Nodes ($N$)')
    ax1.set_ylabel('Communication Time (Seconds)')
    ax1.set_title(f'Scaling Comparison (Model: {M} GB, BW: {B} GB/s)')
    ax1.legend(fontsize=9)
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # --- Right Plot: Bandwidth Efficiency ---
    # PS: Only 1 node (the server) receives at full bandwidth → each worker
    # effectively gets B/N bandwidth
    efficiency_ps = B / N_vals
    
    # Ring: Every node sends and receives simultaneously at full bandwidth B
    efficiency_ring = np.ones_like(N_vals) * B
    
    ax2.plot(N_vals, efficiency_ps, label='PS: Effective BW per Worker', color='#e74c3c', linewidth=2.5)
    ax2.plot(N_vals, efficiency_ring, label='Ring: Effective BW per Worker', color='#2ecc71', linewidth=2.5)
    
    ax2.set_xlim(2, N_max)
    ax2.set_xlabel('Number of Nodes ($N$)')
    ax2.set_ylabel('Effective Bandwidth per Worker (GB/s)')
    ax2.set_title('Network Topology Bandwidth Saturation')
    ax2.legend(fontsize=9)
    ax2.grid(True, linestyle='--', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# === Interactive Widget Setup ===
# Each slider controls one parameter of the simulation
print("Interactive Distributed Training Simulator")
print("=" * 50)
ui = widgets.interactive(
    plot_advanced_communication,
    N_max=widgets.IntSlider(min=2, max=30000, step=100, value=1000,
                             description='Max Nodes:'),
    M=widgets.FloatSlider(min=0.1, max=100.0, step=0.1, value=10.0,
                           description='Model (GB):'),
    B=widgets.FloatSlider(min=10.0, max=1000.0, step=10.0, value=100.0,
                           description='BW (GB/s):')
)
display(ui)

---

### Exercise
1. Set **Max Nodes = 30,000** and observe both graphs. Why does the Parameter Server curve shoot upward while Ring-AllReduce stays flat?

2. Try **Model Size = 100 GB** (equivalent to a large language model). What happens to the communication time? Is Ring-AllReduce still practical?

3. Increase **Bandwidth to 900 GB/s** (simulating NVLink 4.0). How does this change the feasibility of training massive models?

4. In the right graph ("Bandwidth Saturation"), explain in your own words why the Parameter Server effective bandwidth approaches zero as $N$ increases.

**Write your analysis below:**

**(Student explanation here)**